In [1]:
pip install unsloth vllm

In [2]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth vllm
else:
    # [NOTE] Do the below ONLY in Colab! Use [[pip install unsloth vllm]]
    !pip install --no-deps unsloth vllm
# Install latest Hugging Face for Llama-3.1
!pip install --no-deps git+https://github.com/huggingface/transformers@v4.49.0

In [4]:
#@title Colab Extra Install { display-mode: "form" }
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth vllm
else:
    !pip install --no-deps unsloth vllm
    # [NOTE] Do the below ONLY in Colab! Use [[pip install unsloth vllm]]
    # Skip restarting message in Colab
    import sys, re, requests; modules = list(sys.modules.keys())
    for x in modules: sys.modules.pop(x) if "PIL" in x or "google" in x else None
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft "trl==0.15.2" triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf datasets huggingface_hub hf_transfer

    # vLLM requirements - vLLM breaks Colab due to reinstalling numpy
    f = requests.get("https://raw.githubusercontent.com/vllm-project/vllm/refs/heads/main/requirements/common.txt").content
    with open("vllm_requirements.txt", "wb") as file:
        file.write(re.sub(rb"(transformers|numpy|xformers)[^\n]{1,}\n", b"", f))
    !pip install -r vllm_requirements.txt

In [5]:
from google.colab import drive
drive.mount('/content/drive')

from google.colab import files
dataset = files.upload()

dataset_path = next(iter(dataset))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Saving simplification_final_dataset.csv to simplification_final_dataset (4).csv


In [6]:
from unsloth import FastModel
import torch

fourbit_models = [
    # LLaMA 3 models
    "unsloth/Llama-3.1-8B",  # Specifically for Llama-3.1-8B
    "unsloth/Llama-3.2-3B",
    "unsloth/Llama-3.3-70B",

    # Other popular models!
    "unsloth/mistral-7b-instruct-v0.3",
    "unsloth/Phi-4",
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/Llama-3.1-8B",  # Using Llama-3.1-8B model
    max_seq_length = 10000,                # Max sequence length for longer context
    load_in_4bit = True,                  # 4-bit quantization for reduced memory usage
    load_in_8bit = False,                 # Keep as False for better accuracy and memory use
    full_finetuning = False,              # If you need finetuning, set it True
    # token = "hf_...",                   # Use this token for gated models if needed
)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
INFO 04-20 06:48:33 [__init__.py:239] Automatically detected platform cuda.
==((====))==  Unsloth 2025.3.19: Fast Llama patching. Transformers: 4.49.0. vLLM: 0.8.4.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [7]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False, # Turn off for just text!
    finetune_language_layers   = True,  # Should leave on!
    finetune_attention_modules = True,  # Attention good for GRPO
    finetune_mlp_modules       = True,  # SHould leave on always!

    r = 8,           # Larger = higher accuracy, but might overfit
    lora_alpha = 8,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)

Unsloth: Making `model.base_model.model.model` require gradients


In [10]:
# ✅ Load Dataset
from datasets import load_dataset
dataset = load_dataset("csv", data_files=dataset_path)
train_test_split = dataset["train"].train_test_split(test_size=0.04, seed=42)
train_dataset = train_test_split["train"]
test_dataset = train_test_split["test"]
print(f"Train dataset size: {len(train_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")
dataset=train_dataset


# ✅ Format dataset using Chat Template
def format_chat(example):
    return {
        "messages": [
            {"role": "user", "content": f"You are an expert in legal language and its interpretation. Your task is to simplify the following legal text while maintaining its original meaning and intent, so that a layman person can understand. The simplified version should be accessible to individuals without a legal background, using clear and concise language. Simplify the following legal text:\n{example['legal_text']}"},
            {"role": "assistant", "content": f'Simplified Explanation: {example["simplified_text"]}'},
        ]
    }

dataset = dataset.map(format_chat)

Train dataset size: 2040
Test dataset size: 86


Map:   0%|          | 0/2040 [00:00<?, ? examples/s]

In [11]:
dataset[1]['messages']




[{'content': 'You are an expert in legal language and its interpretation. Your task is to simplify the following legal text while maintaining its original meaning and intent, so that a layman person can understand. The simplified version should be accessible to individuals without a legal background, using clear and concise language. Simplify the following legal text:\n131. No Magistrate or police officer shall be compelled to say when he got any\ninformation as to the commission of any offence, and no revenue officer shall be compelled\nto say when he got any information as to the commission of any offence against the public\nrevenue.\nExplanation.—"revenue officer" means any officer employed in or about the business\nof any branch of the public revenue.',
  'role': 'user'},
 {'content': 'Simplified Explanation: Magistrate, police officers, and revenue officers do not have to reveal when they first learned about a crime.',
  'role': 'assistant'}]

In [12]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3",
)

In [13]:
from unsloth.chat_templates import standardize_data_formats
dataset = standardize_data_formats(dataset)

In [14]:
dataset[100]

{'legal_text': "Public Law 117-142\n117th Congress\n\n                                 An Act\n\n\n \nTo designate the facility of the United States Postal Service located at \n6223 Maple Street, in Omaha, Nebraska, as the ``Petty Officer 1st Class \n  Charles Jackson French Post Office''. <<NOTE: June 15, 2022 -  [H.R. \n                                4168]>> \n\n    Be it enacted by the Senate and House of Representatives of the \nUnited States of America in Congress assembled,\nSECTION 1. PETTY OFFICER 1ST CLASS CHARLES JACKSON FRENCH POST \n                              OFFICE.\n\n    (a) Designation.--The facility of the United States Postal Service \nlocated at 6223 Maple Street, in Omaha, Nebraska, shall be known and \ndesignated as the ``Petty Officer 1st Class Charles Jackson French Post \nOffice''.\n    (b) References.--Any reference in a law, map, regulation, document, \npaper, or other record of the United States to the facility referred to \nin subsection (a) shall be dee

In [15]:
def apply_chat_template(examples):
    texts = []
    for message in examples["messages"]:
        if isinstance(message, dict):
            text = message.get('content', '')  # Safely extract 'content' if it exists
        else:
            text = str(message)  # If it's not a dictionary, convert it to a string

        texts.append(text)

    return {"text": texts}
pass
dataset = dataset.map(apply_chat_template, batched = True)

Map:   0%|          | 0/2040 [00:00<?, ? examples/s]

In [16]:
dataset[100]["text"]


'[{\'content\': "You are an expert in legal language and its interpretation. Your task is to simplify the following legal text while maintaining its original meaning and intent, so that a layman person can understand. The simplified version should be accessible to individuals without a legal background, using clear and concise language. Simplify the following legal text:\\nPublic Law 117-142\\n117th Congress\\n\\n                                 An Act\\n\\n\\n \\nTo designate the facility of the United States Postal Service located at \\n6223 Maple Street, in Omaha, Nebraska, as the ``Petty Officer 1st Class \\n  Charles Jackson French Post Office\'\'. <<NOTE: June 15, 2022 -  [H.R. \\n                                4168]>> \\n\\n    Be it enacted by the Senate and House of Representatives of the \\nUnited States of America in Congress assembled,\\nSECTION 1. PETTY OFFICER 1ST CLASS CHARLES JACKSON FRENCH POST \\n                              OFFICE.\\n\\n    (a) Designation.--The fa

In [17]:
def convert_to_gemma_format(examples):
    conversations = []
    for conversation in examples["messages"]:
        formatted_conversation = ""

        for message in conversation:
            role = message["role"]
            content = message["content"]

            # Format the content based on role (user or assistant)
            if role == "user":
                formatted_conversation += f"<start_of_turn>user\n{content}<end_of_turn>\n"
            elif role == "assistant":
                formatted_conversation += f"<start_of_turn>model\n{content}<end_of_turn>\n"

        # Append the formatted conversation to the list
        conversations.append(formatted_conversation)

    return {"text": conversations}

# Applying the transformation function to your dataset
dataset = dataset.map(convert_to_gemma_format, batched=True)


Map:   0%|          | 0/2040 [00:00<?, ? examples/s]

In [18]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    eval_dataset = None, # Can set up evaluation!
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4, # Use GA to mimic batch size!
        warmup_steps = 5,
        num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 30,
        learning_rate = 2e-4, # Reduce to 2e-5 for long training runs
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none", # Use this for WandB etc
        auto_find_batch_size= True
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/2040 [00:00<?, ? examples/s]

In [19]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<start_of_turn>user\n",
    response_part = "<start_of_turn>model\n",
)

Map (num_proc=2):   0%|          | 0/2040 [00:00<?, ? examples/s]

In [20]:
tokenizer.decode(trainer.train_dataset[100]["input_ids"])

'<|begin_of_text|><start_of_turn>user\nYou are an expert in legal language and its interpretation. Your task is to simplify the following legal text while maintaining its original meaning and intent, so that a layman person can understand. The simplified version should be accessible to individuals without a legal background, using clear and concise language. Simplify the following legal text:\nPublic Law 117-142\n117th Congress\n\n                                 An Act\n\n\n \nTo designate the facility of the United States Postal Service located at \n6223 Maple Street, in Omaha, Nebraska, as the ``Petty Officer 1st Class \n  Charles Jackson French Post Office\'\'. <<NOTE: June 15, 2022 -  [H.R. \n                                4168]>> \n\n    Be it enacted by the Senate and House of Representatives of the \nUnited States of America in Congress assembled,\nSECTION 1. PETTY OFFICER 1ST CLASS CHARLES JACKSON FRENCH POST \n                              OFFICE.\n\n    (a) Designation.--Th

In [21]:
tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[100]["labels"]]).replace(tokenizer.pad_token, " ")

'                                                                                                                                                                                                                                                                                                                                          Simplified Explanation: Renaming a Post Office: The law renames the United States Postal Service facility at 6223 Maple Street in Omaha, Nebraska to the "Petty Officer 1st Class Charles Jackson French Post Office." Any official references to the facility will be changed to reflect the new name.<end_of_turn>\n'

In [22]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.741 GB.
5.67 GB of memory reserved.


In [23]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,040 | Num Epochs = 1 | Total steps = 30
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 20,971,520/8,000,000,000 (0.26% trained)
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,0.890500
2,0.965900
3,1.002900
4,0.804000
5,0.774800
6,0.680500
7,0.717600
8,0.670300
9,0.641200
10,0.701400


In [24]:
model.save_pretrained("/content/drive/MyDrive/legal_nlp/llama_legal_simplifier")
tokenizer.save_pretrained("/content/drive/MyDrive/legal_nlp/llama_legal_simplifier")

('/content/drive/MyDrive/legal_nlp/llama_legal_simplifier/tokenizer_config.json',
 '/content/drive/MyDrive/legal_nlp/llama_legal_simplifier/special_tokens_map.json',
 '/content/drive/MyDrive/legal_nlp/llama_legal_simplifier/tokenizer.json')

In [25]:
import gc
tensor = None
gc.collect()
torch.cuda.empty_cache()


In [26]:
print(torch.cuda.memory_summary(device=None, abbreviated=False))

|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |   5871 MiB |   7014 MiB |   9259 GiB |   9253 GiB |
|       from large pool |   5646 MiB |   6743 MiB |   9197 GiB |   9191 GiB |
|       from small pool |    224 MiB |    309 MiB |     61 GiB |     61 GiB |
|---------------------------------------------------------------------------|
| Active memory         |   5871 MiB |   7014 MiB |   9259 GiB |   9253 GiB |
|       from large pool |   5646 MiB |   6743 MiB |   9197 GiB |

In [27]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

487.0331 seconds used for training.
8.12 minutes used for training.
Peak reserved memory = 7.027 GB.
Peak reserved memory for training = 1.357 GB.
Peak reserved memory % of max memory = 47.67 %.
Peak reserved memory for training % of max memory = 9.206 %.


In [28]:
!pip install unsloth evaluate bert_score rouge_score

In [29]:
from unsloth import FastLanguageModel
import evaluate
from transformers import pipeline


model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "/content/drive/MyDrive/legal_nlp/llama_legal_simplifier",  # Path to your fine-tuned model
    dtype = torch.float16,
    load_in_4bit = True,
    use_safetensors = True,
    device_map = "auto"
)

# ✅ Enable inference optimizations
FastLanguageModel.for_inference(model)


==((====))==  Unsloth 2025.3.19: Fast Llama patching. Transformers: 4.49.0. vLLM: 0.8.4.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth 2025.3.19 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 4096, padding_idx=128004)
        (layers): ModuleList(
          (0): LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear4b

In [49]:
# ✅ Load a small test subset
test_dataset = train_test_split["test"]

In [75]:
from transformers import AutoTokenizer

# ✅ Create generation prompts using chat template
def make_chat_prompt(legal_text):
    # The prompt instructs the model to simplify the legal text while keeping the meaning intact.
    prompt = f"""
    You are an expert in legal language and its interpretation. Your task is to simplify the following legal text while maintaining its original meaning and intent. The simplified version should be accessible to individuals without a legal background, using clear and concise language. Simplify the following legal text:

    {legal_text}
    """
    return prompt



In [79]:
# ✅ Create generation prompts using chat template
def make_chat_prompt(legal_text):
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": f"You are an expert in legal language and its interpretation. Your task is to simplify the following legal text while maintaining its original meaning and intent, so that a layman person can understand. The simplified version should be accessible to individuals without a legal background, using clear and concise language. Simplify the following legal text:\n{example['legal_text']}"}],
        tokenize=False,
        add_generation_prompt=True
    )

predictions, references = [], []

def simplify_text(legal_text):
    prompt = make_chat_prompt(legal_text)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=8192,
            do_sample=False,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1
        )
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return decoded

In [80]:
for example in test_dataset:
    decoded = simplify_text(example["legal_text"])
    # Remove the prompt portion to get only the response
    simplified_output = decoded.split("\nmodel\n")[-1].strip()
    predictions.append(simplified_output.strip())
    references.append(example["simplified_text"].strip())


ValueError: The following `model_kwargs` are not used by the model: ['num_logits_to_keep'] (note: typos in the generate arguments will also show up in this list)

In [81]:

# ✅ Evaluate with ROUGE, BLEU, BERTScore
rouge = evaluate.load("rouge")
bertscore = evaluate.load("bertscore")


In [82]:
rouge_score = rouge.compute(predictions=predictions, references=references)
bertscore_score = bertscore.compute(predictions=predictions, references=references, lang="en")
print("ROUGE Scores:", rouge_score)
print("BERTScore Scores:")
for metric, value in bertscore_score.items():
    try:
      print(f"{metric}: {sum(value)/len(value)}")
    except TypeError:
      print(f"{metric}: {value}")

import random
random_index = random.randint(0, len(predictions) - 1)
print(f"Random Index: {random_index}")
print(f"\n\n\nPrediction: {predictions[random_index]}")

print(f"\n\n\nReference: {references[random_index]}")

IndexError: list index out of range